In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── 0. Style ──────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['figure.dpi'] = 150

# ── 1. Load data ──────────────────────────────────────────────────────────────
micro = pd.read_csv("micro_clean.csv")
large = pd.read_csv("large_clean.csv")

print(f"Micro: {len(micro)} rows | Large: {len(large)} rows")

# ── 2. Align day-of-week columns ──────────────────────────────────────────────
day_map = {0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday',
           4:'Friday', 5:'Saturday', 6:'Sunday'}
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

# micro already has string 'post_day'; large has integer 'publish_weekday'
large['post_day'] = large['publish_weekday'].map(day_map)

print("Day columns aligned.")

# ── 3. Helper ─────────────────────────────────────────────────────────────────
def save(fig, name):
    fig.tight_layout()
    fig.savefig(name, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {name}")

# ══════════════════════════════════════════════════════════════════════════════
# CHART 1 — Median views by publish hour (micro vs large)
# ══════════════════════════════════════════════════════════════════════════════
micro_hour = micro.groupby('publish_hour')['view_count'].median()
large_hour = large.groupby('publish_hour')['view_count'].median()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(micro_hour.index, micro_hour.values, marker='o', label='Micro channels', color='#534AB7', linewidth=2)
ax.plot(large_hour.index, large_hour.values, marker='s', label='Large channels', color='#1D9E75', linewidth=2)
ax.set_title("Median view count by publish hour")
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("Median views")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xticks(range(0, 24))
ax.legend()
save(fig, "chart1_views_by_hour.png")

# ══════════════════════════════════════════════════════════════════════════════
# CHART 2 — Median like rate by publish hour
# ══════════════════════════════════════════════════════════════════════════════
micro_hr_lr = micro.groupby('publish_hour')['like_rate'].median()
large_hr_lr = large.groupby('publish_hour')['like_rate'].median()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(micro_hr_lr.index, micro_hr_lr.values, marker='o', label='Micro channels', color='#534AB7', linewidth=2)
ax.plot(large_hr_lr.index, large_hr_lr.values, marker='s', label='Large channels', color='#1D9E75', linewidth=2)
ax.set_title("Median like rate by publish hour")
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("Median like rate")
ax.set_xticks(range(0, 24))
ax.legend()
save(fig, "chart2_likerate_by_hour.png")

# ══════════════════════════════════════════════════════════════════════════════
# CHART 3 — Median views by day of week
# ══════════════════════════════════════════════════════════════════════════════
micro_day = micro.groupby('post_day')['view_count'].median().reindex(day_order)
large_day = large.groupby('post_day')['view_count'].median().reindex(day_order)

x = np.arange(len(day_order))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, micro_day.values, width, label='Micro', color='#534AB7', alpha=0.85)
ax.bar(x + width/2, large_day.values, width, label='Large', color='#1D9E75', alpha=0.85)
ax.set_title("Median view count by day of week")
ax.set_xticks(x)
ax.set_xticklabels(day_order, rotation=15)
ax.set_ylabel("Median views")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
save(fig, "chart3_views_by_day.png")

# ══════════════════════════════════════════════════════════════════════════════
# CHART 4 & 5 — Posting heatmaps (micro and large)
# ══════════════════════════════════════════════════════════════════════════════
def make_heatmap(df, title, filename, cmap='YlOrRd'):
    pivot = df.pivot_table(
        values='view_count',
        index='post_day',
        columns='publish_hour',
        aggfunc='median'
    ).reindex(day_order)

    # Fill any missing hour slots with 0
    pivot = pivot.reindex(columns=range(24), fill_value=0)

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(
        pivot, ax=ax, cmap=cmap, linewidths=0.3,
        fmt='.0f', annot=False,
        cbar_kws={'label': 'Median views'}
    )
    ax.set_title(title)
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_ylabel("")
    save(fig, filename)
    return pivot

micro_pivot = make_heatmap(micro, "Posting heatmap — Micro channels", "chart4_heatmap_micro.png", cmap='Purples')
large_pivot = make_heatmap(large, "Posting heatmap — Large channels", "chart5_heatmap_large.png", cmap='Greens')

# ══════════════════════════════════════════════════════════════════════════════
# CHART 6 — Gap heatmap (large minus micro, normalized)
# ══════════════════════════════════════════════════════════════════════════════
# Normalize each pivot to 0-1 so scale differences don't dominate
def normalize(df):
    return (df - df.min().min()) / (df.max().max() - df.min().min() + 1e-9)

micro_norm = normalize(micro_pivot)
large_norm = normalize(large_pivot)
gap = large_norm - micro_norm  # positive = large dominates; negative = micro opportunity

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    gap, ax=ax, cmap='RdYlGn_r', center=0, linewidths=0.3,
    cbar_kws={'label': 'Large advantage  →  Micro opportunity'}
)
ax.set_title("Gap heatmap — Green = micro opportunity (large channels post less / perform less here)")
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("")
save(fig, "chart6_gap_heatmap.png")

# ══════════════════════════════════════════════════════════════════════════════
# CHART 7 — Video duration analysis
# ══════════════════════════════════════════════════════════════════════════════
bins   = [0, 60, 300, 600, 1800, 99999]
labels = ['≤1 min', '1–5 min', '5–10 min', '10–30 min', '30+ min']

micro['duration_bucket_std'] = pd.cut(micro['duration_sec'], bins=bins, labels=labels)
large['duration_bucket_std'] = pd.cut(large['duration_sec'],  bins=bins, labels=labels)

micro_dur = micro.groupby('duration_bucket_std', observed=True)['view_count'].median()
large_dur = large.groupby('duration_bucket_std', observed=True)['view_count'].median()

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, micro_dur.reindex(labels).values, width, label='Micro', color='#534AB7', alpha=0.85)
ax.bar(x + width/2, large_dur.reindex(labels).values, width, label='Large', color='#1D9E75', alpha=0.85)
ax.set_title("Median view count by video duration")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Median views")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
save(fig, "chart7_views_by_duration.png")

# ══════════════════════════════════════════════════════════════════════════════
# GOLDEN WINDOWS — Top 10 time slots for micro creators
# ══════════════════════════════════════════════════════════════════════════════

# Micro: rank each (day, hour) cell by median view count (percentile)
micro_perf = micro.groupby(['post_day', 'publish_hour'])['view_count'].median().reset_index()
micro_perf.columns = ['day', 'hour', 'micro_median_views']
micro_perf['micro_rank'] = micro_perf['micro_median_views'].rank(pct=True)

# Large: count how many large videos were posted at each (day, hour)
large_density = large.groupby(['post_day', 'publish_hour']).size().reset_index()
large_density.columns = ['day', 'hour', 'large_video_count']
large_density['large_density_rank'] = large_density['large_video_count'].rank(pct=True)

# Merge
golden = pd.merge(micro_perf, large_density, on=['day', 'hour'], how='left')
golden['large_video_count'] = golden['large_video_count'].fillna(0)
golden['large_density_rank'] = golden['large_density_rank'].fillna(0)

# Golden score: high micro performance + low large density
golden['golden_score'] = golden['micro_rank'] - golden['large_density_rank']
golden = golden.sort_values('golden_score', ascending=False)

# Reorder day column
golden['day'] = pd.Categorical(golden['day'], categories=day_order, ordered=True)
top10 = golden.head(10).sort_values(['day', 'hour'])

print("\n═══════════════════════════════════════════")
print("  TOP 10 GOLDEN POSTING WINDOWS FOR MICRO CREATORS")
print("═══════════════════════════════════════════")
print(top10[['day', 'hour', 'micro_median_views', 'large_video_count', 'golden_score']].to_string(index=False))

# Save golden windows as CSV
top10.to_csv("golden_windows.csv", index=False)
print("\nSaved: golden_windows.csv")

# ══════════════════════════════════════════════════════════════════════════════
# CHART 8 — Golden windows bar chart
# ══════════════════════════════════════════════════════════════════════════════
top10['label'] = top10['day'].astype(str) + ' ' + top10['hour'].astype(str) + ':00'

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top10['label'], top10['micro_median_views'], color='#534AB7', alpha=0.85)
ax.set_title("Top 10 Golden Posting Windows — Micro creator median views")
ax.set_xlabel("Median view count")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.invert_yaxis()
save(fig, "chart8_golden_windows.png")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
best_hour_micro  = int(micro_hour.idxmax())
best_day_micro   = micro_day.idxmax()
best_duration    = micro_dur.idxmax()
top_window       = top10.iloc[0]

print("\n═══════════════════════════════════════════")
print("  ACTIONABLE ROADMAP SUMMARY")
print("═══════════════════════════════════════════")
print(f"  Best hour to post (micro):     {best_hour_micro}:00 UTC")
print(f"  Best day to post (micro):      {best_day_micro}")
print(f"  Best video duration (micro):   {best_duration}")
print(f"  #1 Golden Window:              {top_window['day']} at {int(top_window['hour'])}:00 UTC")
print(f"     → Micro median views:       {int(top_window['micro_median_views']):,}")
print(f"     → Large channel posts:      {int(top_window['large_video_count'])}")
print("═══════════════════════════════════════════")


Micro: 1035 rows | Large: 1177 rows
Day columns aligned.
Saved: chart1_views_by_hour.png
Saved: chart2_likerate_by_hour.png
Saved: chart3_views_by_day.png
Saved: chart4_heatmap_micro.png
Saved: chart5_heatmap_large.png
Saved: chart6_gap_heatmap.png
Saved: chart7_views_by_duration.png

═══════════════════════════════════════════
  TOP 10 GOLDEN POSTING WINDOWS FOR MICRO CREATORS
═══════════════════════════════════════════
     day  hour  micro_median_views  large_video_count  golden_score
  Monday     4              1261.0                2.0      0.683305
  Monday     6              1940.0                3.0      0.681768
  Monday     7              1191.0                2.0      0.601672
  Monday     9              1283.0                1.0      0.782026
  Monday    11              1294.0                3.0      0.620543
  Friday     5              5726.0                4.0      0.626199
  Friday    11              7233.5                4.0      0.636403
Saturday     9            14383

In [1]:
"""
Phase III Extension: Golden Posting Windows by Content Type
============================================================
Requirements: pandas, matplotlib, seaborn
Install:  pip install pandas matplotlib seaborn

Put micro_clean.csv and large_clean.csv in the same folder, then run:
    python phase3_golden_windows_by_content.py

Output: one heatmap + golden window table per content dimension
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams['figure.dpi'] = 150

os.makedirs("content_golden_windows", exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────────
micro = pd.read_csv("micro_clean.csv")
large = pd.read_csv("large_clean.csv")

# Align day column for large
day_map = {0:'Monday', 1:'Tuesday', 2:'Wednesday', 3:'Thursday',
           4:'Friday', 5:'Saturday', 6:'Sunday'}
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
large['post_day'] = large['publish_weekday'].map(day_map)

print(f"Loaded — Micro: {len(micro)} rows | Large: {len(large)} rows\n")

# ── Core helpers ──────────────────────────────────────────────────────────────

def get_large_density(large_subset=None):
    """Count how many large-channel videos were posted at each (day, hour)."""
    df = large if large_subset is None else large_subset
    density = df.groupby(['post_day', 'publish_hour']).size().reset_index()
    density.columns = ['day', 'hour', 'large_count']
    return density


def golden_windows(micro_subset, large_subset=None, top_n=5):
    """
    Compute golden posting windows for a micro subset.
    Returns a DataFrame of top_n slots ranked by:
        high micro performance + low large channel density
    """
    if len(micro_subset) < 10:
        return None

    # Micro: median views per (day, hour)
    perf = (micro_subset
            .groupby(['post_day', 'publish_hour'])['view_count']
            .median()
            .reset_index())
    perf.columns = ['day', 'hour', 'micro_median_views']
    perf['micro_rank'] = perf['micro_median_views'].rank(pct=True)

    # Large: density per (day, hour)
    density = get_large_density(large_subset)
    density['large_density_rank'] = density['large_count'].rank(pct=True)

    merged = pd.merge(perf, density, on=['day', 'hour'], how='left')
    merged['large_count'] = merged['large_count'].fillna(0)
    merged['large_density_rank'] = merged['large_density_rank'].fillna(0)

    merged['golden_score'] = merged['micro_rank'] - merged['large_density_rank']
    merged = merged.sort_values('golden_score', ascending=False)
    merged['day'] = pd.Categorical(merged['day'], categories=day_order, ordered=True)

    return merged.head(top_n)


def plot_heatmap(micro_subset, title, filename):
    """Plot a posting heatmap for a micro subset."""
    if len(micro_subset) < 10:
        print(f"  Skipping heatmap (too few rows): {title}")
        return

    pivot = (micro_subset
             .pivot_table(values='view_count', index='post_day',
                          columns='publish_hour', aggfunc='median')
             .reindex(day_order)
             .reindex(columns=range(24), fill_value=0))

    fig, ax = plt.subplots(figsize=(16, 4))
    sns.heatmap(pivot, ax=ax, cmap='Purples', linewidths=0.3,
                cbar_kws={'label': 'Median views'})
    ax.set_title(title)
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {filename}")


def print_windows(windows_df, label):
    """Print golden windows summary table to terminal."""
    if windows_df is None:
        print(f"  [{label}] Not enough data to compute windows.\n")
        return
    print(f"\n  ── {label} ──")
    print(f"  {'Day':<12} {'Hour':>5}  {'Micro Views':>12}  {'Large Posts':>11}  {'Score':>7}")
    print(f"  {'-'*55}")
    for _, r in windows_df.iterrows():
        print(f"  {str(r['day']):<12} {int(r['hour']):>5}:00  "
              f"{int(r['micro_median_views']):>12,}  "
              f"{int(r['large_count']):>11}  "
              f"{r['golden_score']:>7.3f}")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# DIMENSION 1 — Content Type (Vlog / Short / Daily Life)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("DIMENSION 1: Content Type")
print("=" * 60)

content_types = {
    'Vlog':        micro[micro['is_vlog'] == 1],
    'Short':       micro[micro['is_short'] == 1],
    'Daily Life':  micro[micro['is_daily_life'] == 1],
    'Regular':     micro[(micro['is_vlog'] == 0) &
                         (micro['is_short'] == 0) &
                         (micro['is_daily_life'] == 0)],
}

all_windows_d1 = []
for label, subset in content_types.items():
    print(f"\n[{label}] — {len(subset)} videos")
    plot_heatmap(
        subset,
        f"Posting heatmap — {label} videos (micro channels)",
        f"content_golden_windows/heatmap_type_{label.replace(' ','_').lower()}.png"
    )
    windows = golden_windows(subset)
    print_windows(windows, label)
    if windows is not None:
        windows.insert(0, 'content_type', label)
        all_windows_d1.append(windows)

if all_windows_d1:
    pd.concat(all_windows_d1).to_csv(
        "content_golden_windows/golden_windows_by_content_type.csv", index=False)
    print("Saved: content_golden_windows/golden_windows_by_content_type.csv")


# ══════════════════════════════════════════════════════════════════════════════
# DIMENSION 2 — Category (top 5 in micro)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("DIMENSION 2: Category")
print("=" * 60)

top_cats = micro['category_name'].value_counts().head(5).index.tolist()
all_windows_d2 = []

for cat in top_cats:
    micro_cat = micro[micro['category_name'] == cat]
    large_cat = large[large['category_name'] == cat]
    print(f"\n[{cat}] — micro: {len(micro_cat)} | large: {len(large_cat)}")

    plot_heatmap(
        micro_cat,
        f"Posting heatmap — {cat} (micro channels)",
        f"content_golden_windows/heatmap_cat_{cat.replace(' ','_').replace('&','and').lower()}.png"
    )
    windows = golden_windows(micro_cat, large_cat if len(large_cat) > 5 else None)
    print_windows(windows, cat)
    if windows is not None:
        windows.insert(0, 'category', cat)
        all_windows_d2.append(windows)

if all_windows_d2:
    pd.concat(all_windows_d2).to_csv(
        "content_golden_windows/golden_windows_by_category.csv", index=False)
    print("Saved: content_golden_windows/golden_windows_by_category.csv")


# ══════════════════════════════════════════════════════════════════════════════
# DIMENSION 3 — Video Length
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("DIMENSION 3: Video Length")
print("=" * 60)

# Standardize duration buckets
bins   = [0, 60, 600, 99999]
labels_dur = ['Short (≤1 min)', 'Mid (1–10 min)', 'Long (10+ min)']

micro['dur_group'] = pd.cut(micro['duration_sec'], bins=bins, labels=labels_dur)
large['dur_group'] = pd.cut(large['duration_sec'],  bins=bins, labels=labels_dur)

all_windows_d3 = []
for dur in labels_dur:
    micro_dur = micro[micro['dur_group'] == dur]
    large_dur = large[large['dur_group'] == dur]
    print(f"\n[{dur}] — micro: {len(micro_dur)} | large: {len(large_dur)}")

    plot_heatmap(
        micro_dur,
        f"Posting heatmap — {dur} videos (micro channels)",
        f"content_golden_windows/heatmap_dur_{dur.split(' ')[0].lower()}.png"
    )
    windows = golden_windows(micro_dur, large_dur if len(large_dur) > 5 else None)
    print_windows(windows, dur)
    if windows is not None:
        windows.insert(0, 'duration_group', dur)
        all_windows_d3.append(windows)

if all_windows_d3:
    pd.concat(all_windows_d3).to_csv(
        "content_golden_windows/golden_windows_by_duration.csv", index=False)
    print("Saved: content_golden_windows/golden_windows_by_duration.csv")


# ══════════════════════════════════════════════════════════════════════════════
# DIMENSION 4 — First Video vs Established Micro Channels
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("DIMENSION 4: First Video vs Established")
print("=" * 60)

channel_types = {
    'First video (brand new)': micro[micro['is_first_video'] == 1],
    'Established micro':       micro[micro['is_first_video'] == 0],
}

all_windows_d4 = []
for label, subset in channel_types.items():
    print(f"\n[{label}] — {len(subset)} videos")
    plot_heatmap(
        subset,
        f"Posting heatmap — {label}",
        f"content_golden_windows/heatmap_channel_{label.split(' ')[0].lower()}.png"
    )
    windows = golden_windows(subset)
    print_windows(windows, label)
    if windows is not None:
        windows.insert(0, 'channel_type', label)
        all_windows_d4.append(windows)

if all_windows_d4:
    pd.concat(all_windows_d4).to_csv(
        "content_golden_windows/golden_windows_by_channel_type.csv", index=False)
    print("Saved: content_golden_windows/golden_windows_by_channel_type.csv")


# ══════════════════════════════════════════════════════════════════════════════
# BONUS — Side-by-side comparison heatmap (top 4 content types)
# ══════════════════════════════════════════════════════════════════════════════
print("\nGenerating comparison heatmap...")

fig, axes = plt.subplots(2, 2, figsize=(20, 10))
axes = axes.flatten()

plot_configs = [
    ('Vlog',       micro[micro['is_vlog'] == 1]),
    ('Short',      micro[micro['is_short'] == 1]),
    ('Daily Life', micro[micro['is_daily_life'] == 1]),
    ('Regular',    micro[(micro['is_vlog']==0)&(micro['is_short']==0)&(micro['is_daily_life']==0)]),
]

for ax, (label, subset) in zip(axes, plot_configs):
    if len(subset) < 10:
        ax.set_visible(False)
        continue
    pivot = (subset
             .pivot_table(values='view_count', index='post_day',
                          columns='publish_hour', aggfunc='median')
             .reindex(day_order)
             .reindex(columns=range(24), fill_value=0))
    sns.heatmap(pivot, ax=ax, cmap='Purples', linewidths=0.2,
                cbar_kws={'label': 'Median views'}, xticklabels=2)
    ax.set_title(f"{label}  (n={len(subset)})", fontsize=12)
    ax.set_xlabel("Hour (UTC)")
    ax.set_ylabel("")

fig.suptitle("Golden Posting Windows by Content Type — Micro Channels", fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig("content_golden_windows/comparison_all_content_types.png", bbox_inches='tight')
plt.close(fig)
print("Saved: content_golden_windows/comparison_all_content_types.png")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  ALL DONE")
print("=" * 60)
print("Output folder: content_golden_windows/")
print()
print("  Heatmaps:")
print("    heatmap_type_vlog.png")
print("    heatmap_type_short.png")
print("    heatmap_type_daily_life.png")
print("    heatmap_type_regular.png")
print("    heatmap_cat_*.png  (one per top category)")
print("    heatmap_dur_*.png  (short / mid / long)")
print("    heatmap_channel_first.png")
print("    heatmap_channel_established.png")
print("    comparison_all_content_types.png")
print()
print("  Golden window tables (CSV):")
print("    golden_windows_by_content_type.csv")
print("    golden_windows_by_category.csv")
print("    golden_windows_by_duration.csv")
print("    golden_windows_by_channel_type.csv")

Loaded — Micro: 1035 rows | Large: 1177 rows

DIMENSION 1: Content Type

[Vlog] — 526 videos
  Saved: content_golden_windows/heatmap_type_vlog.png

  ── Vlog ──
  Day           Hour   Micro Views  Large Posts    Score
  -------------------------------------------------------
  Monday           4:00         1,933            2    0.811
  Monday           9:00         1,259            1    0.759
  Monday           6:00         1,940            3    0.732
  Monday          12:00         1,160            2    0.637
  Monday          11:00         1,294            3    0.616


[Short] — 319 videos
  Saved: content_golden_windows/heatmap_type_short.png

  ── Short ──
  Day           Hour   Micro Views  Large Posts    Score
  -------------------------------------------------------
  Monday           4:00         1,728            2    0.748
  Monday           9:00         1,357            1    0.737
  Monday           6:00         1,991            3    0.685
  Tuesday          6:00         1,37